# Fix: Bonava A och Ratos A
Kör de två cellerna med **Shift + Enter**. Ladda sedan ner `tvilling_priser_lseg_2.csv` och skicka till Claude, plus en skärmdump av utskriften.

In [ ]:
import pandas as pd
try:
    import lseg.data as ld
except Exception:
    import refinitiv.data as ld
ld.open_session()
print('session öppen')

In [ ]:
RICS = ['BONAVa.ST', 'RATOa.ST']
out = []
for r in RICS:
    # metod 1: standardfält (stängningskurs och volym direkt från börsen)
    try:
        h = ld.get_history(universe=r, start='2006-01-01', end='2026-10-31', interval='daily')
        h = h.reset_index(); h['ric'] = r; h['metod'] = 1
        print(r, 'metod 1:', len(h), 'rader, kolumner', list(h.columns)[:6])
        out.append(h)
    except Exception as e:
        print(r, 'metod 1 FEL', str(e)[:120])
    # metod 2: get_data med datum
    try:
        t = ld.get_data(universe=r, fields=['TR.PriceClose.date', 'TR.PriceClose', 'TR.Volume'],
                        parameters={'SDate': '2006-01-01', 'EDate': '2026-10-30', 'Frq': 'D'})
        t['ric'] = r; t['metod'] = 2
        print(r, 'metod 2:', len(t), 'rader, priser:', t.iloc[:, 2].notna().sum())
        out.append(t)
    except Exception as e:
        print(r, 'metod 2 FEL', str(e)[:120])
df = pd.concat(out, ignore_index=True)
df.to_csv('tvilling_priser_lseg_2.csv', index=False)
print('sparat:', len(df), 'rader')